# Basic Chatbot with LangGraph

Build chatbots using LangGraph: from minimal graph workflows to tool-augmented agents with conversation memory and streaming.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

## 1. Defining State and Model

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
from langchain.chat_models import init_chat_model

# Initialize chat model with Google Gemini
llm = init_chat_model("google_genai:gemini-2.5-flash")
llm

## 2. Basic Chatbot Node & Graph

In [ ]:
# Node functionality
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

# Add node
graph_builder.add_node("llmchatbot", chatbot)

# Add edges
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

# Compile graph
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({"messages": "Hi!"})
print(response["messages"][-1].content)

In [ ]:
for event in graph.stream({"messages": "Hi, how are you?"}):
    for value in event.values():
        print(value["messages"][-1].content)

## 3. Chatbot With Tools

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)
tool.invoke("What is LangGraph?")

In [ ]:
# Custom function tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers.

    Args:
        a: First integer
        b: Second integer

    Returns:
        The product of a and b
    """
    return a * b

In [ ]:
tools = [tool, multiply]
llm_with_tool = llm.bind_tools(tools)
llm_with_tool

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

# Node definition
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

# Graph with tools
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

In [ ]:
response = graph.invoke({"messages": "What is the recent AI news?"})
for m in response["messages"]:
    m.pretty_print()

In [ ]:
response = graph.invoke({"messages": "What is 5 multiplied by 2?"})
for m in response["messages"]:
    m.pretty_print()

In [ ]:
response = graph.invoke({"messages": "Give me the recent AI news and then multiply 5 by 10"})
for m in response["messages"]:
    m.pretty_print()

## 4. Adding Memory (Checkpointer)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile(checkpointer=memory)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

response = graph.invoke({"messages": "Hello, my name is Alex and I love coding."}, config=config)
print(response["messages"][-1].content)

In [ ]:
response = graph.invoke({"messages": "What is my name and what do I love?"}, config=config)
print(response["messages"][-1].content)

## 5. Streaming Responses

In [ ]:
config = {"configurable": {"thread_id": "2"}}

for chunk in graph.stream({"messages": "Tell me a short two-line joke"}, config=config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()